#Imports de _librerias_

In [0]:
from pyspark.sql.functions import col, when

# Lectura de la Tabla Bronce olist_products

In [0]:
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_products_dataset")

In [0]:
df.display()

#Transformaciones

In [0]:
df = (
    df
    # Renombrar columnas
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    
    # Tipos de datos
    .withColumn("product_weight_g", col("product_weight_g").cast("int"))
    .withColumn("product_length_cm", col("product_length_cm").cast("int"))
    .withColumn("product_height_cm", col("product_height_cm").cast("int"))
    .withColumn("product_width_cm", col("product_width_cm").cast("int"))

    # Reemplazo de nulos
    .fillna({
        "product_category_name": "unknown",
        "product_photos_qty": 0
    })

    # Filtro de datos inválidos
    .filter(col("product_id").isNotNull())

    # Enriquecimiento: volumen (cm3)
    .withColumn(
        "product_volume_cm3",
        col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
    )

    # Clasificación simple por peso
    .withColumn(
        "weight_category",
        when(col("product_weight_g") < 500, "light")
        .when(col("product_weight_g") < 2000, "medium")
        .otherwise("heavy")
    )

    # Deduplicación
    .dropDuplicates(["product_id"])
)

# Crear la tabla Silver de olist_products

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_products")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_products